# Nested CV 结果分析

这个 notebook 读取统一路径配置中的模型结果目录，生成总体指标汇总和负类误报来源分析。缺少结果时只显示“尚无结果”，不会中断。

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

PROJECT_ROOT = Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if not SCRIPTS_DIR.exists():
    PROJECT_ROOT = Path.cwd().parents[1]
    SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from project_config import BINARY_LABELS_PATH as LABELS_PATH, MODEL_ROOT, PROJECT_ROOT
SUMMARY_PATH = MODEL_ROOT / "nested_cv_summary.csv"
ERROR_SUMMARY_PATH = MODEL_ROOT / "negative_type_error_summary.csv"
METRIC_FIELDS = ["precision_0_5", "recall_0_5", "f1_0_5", "precision_opt", "recall_opt", "f1_opt", "auprc", "auroc"]

print(f"模型结果目录: {MODEL_ROOT}")

In [ ]:
def read_table(path: Path) -> pd.DataFrame:
    if path.is_file():
        return pd.read_csv(path)
    return pd.DataFrame()

metric_rows = []
for outer_dir in sorted(MODEL_ROOT.glob("outer_*")):
    metrics = read_table(outer_dir / "metrics.csv")
    if metrics.empty:
        continue
    row = metrics.iloc[0].to_dict()
    row["fold"] = outer_dir.name.replace("outer_", "")
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
if metrics_df.empty:
    print("尚无 metrics.csv。请先完成训练 notebook。")
else:
    display(metrics_df[["fold", *[field for field in METRIC_FIELDS if field in metrics_df.columns]]])

In [ ]:
if not metrics_df.empty:
    summary_rows = []
    for _, row in metrics_df.iterrows():
        summary_rows.append({"fold": row["fold"], **{field: row.get(field, "") for field in METRIC_FIELDS}})
    summary_rows.append({"fold": "mean", **{field: metrics_df[field].mean() for field in METRIC_FIELDS if field in metrics_df}})
    summary_rows.append({"fold": "std", **{field: metrics_df[field].std(ddof=1) if len(metrics_df) > 1 else 0.0 for field in METRIC_FIELDS if field in metrics_df}})
    summary = pd.DataFrame(summary_rows)
    SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8")
    display(summary)
    print(f"已写出 {SUMMARY_PATH}")

In [ ]:
labels = read_table(LABELS_PATH)
lookup = {}
if not labels.empty and "clip_path" in labels.columns:
    labels = labels.copy()
    labels["file_name"] = labels["clip_path"].map(lambda value: Path(str(value)).name)
    lookup = labels.set_index("file_name").to_dict("index")
print(f"标签索引数量: {len(lookup)}")

In [ ]:
prediction_rows = []
for outer_dir in sorted(MODEL_ROOT.glob("outer_*")):
    predictions = read_table(outer_dir / "predictions.csv")
    metrics = read_table(outer_dir / "metrics.csv")
    if predictions.empty:
        continue
    threshold = float(metrics.iloc[0].get("optimal_threshold", 0.5)) if not metrics.empty else 0.5
    predictions = predictions.copy()
    predictions["outer_fold"] = outer_dir.name.replace("outer_", "")
    predictions["threshold"] = threshold
    prediction_rows.append(predictions)

predictions_df = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
if predictions_df.empty:
    print("尚无 predictions.csv。")
else:
    predictions_df["file_name"] = predictions_df["file"].map(lambda value: Path(str(value)).name)
    predictions_df["score"] = predictions_df["score"].astype(float)
    predictions_df["pred_opt"] = predictions_df["score"] >= predictions_df["threshold"]
    predictions_df["pred_0_5"] = predictions_df["score"] >= 0.5
    meta = predictions_df["file_name"].map(lookup)
    predictions_df["negative_type"] = meta.map(lambda item: item.get("negative_type", "unknown") if isinstance(item, dict) else "unknown")
    predictions_df["source_file"] = meta.map(lambda item: item.get("source_file", "unknown") if isinstance(item, dict) else "unknown")
    predictions_df["species"] = meta.map(lambda item: item.get("species", "unknown") if isinstance(item, dict) and item.get("species") else "unknown")
    display(predictions_df.head())

In [ ]:
if not predictions_df.empty:
    negatives = predictions_df[predictions_df["true_label"].astype(str) == "0"].copy()
    error_summary = (
        negatives.groupby("negative_type")
        .agg(
            negative_samples=("file", "count"),
            false_positives_opt=("pred_opt", "sum"),
            false_positives_0_5=("pred_0_5", "sum"),
        )
        .reset_index()
    )
    error_summary["false_positive_rate_opt"] = error_summary["false_positives_opt"] / error_summary["negative_samples"]
    error_summary["false_positive_rate_0_5"] = error_summary["false_positives_0_5"] / error_summary["negative_samples"]
    ERROR_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    error_summary.to_csv(ERROR_SUMMARY_PATH, index=False, encoding="utf-8")
    display(error_summary)
    print(f"已写出 {ERROR_SUMMARY_PATH}")

In [ ]:
if not predictions_df.empty:
    negatives = predictions_df[predictions_df["true_label"].astype(str) == "0"]
    source_summary = (
        negatives.groupby("source_file")
        .agg(negative_samples=("file", "count"), false_positives_opt=("pred_opt", "sum"))
        .reset_index()
    )
    source_summary["false_positive_rate_opt"] = source_summary["false_positives_opt"] / source_summary["negative_samples"]
    display(source_summary.sort_values(["false_positives_opt", "false_positive_rate_opt"], ascending=False).head(20))

In [ ]:
if not predictions_df.empty:
    negatives = predictions_df[predictions_df["true_label"].astype(str) == "0"]
    species_summary = (
        negatives.groupby("species")
        .agg(negative_samples=("file", "count"), false_positives_opt=("pred_opt", "sum"))
        .reset_index()
    )
    species_summary["false_positive_rate_opt"] = species_summary["false_positives_opt"] / species_summary["negative_samples"]
    display(species_summary.sort_values(["false_positives_opt", "false_positive_rate_opt"], ascending=False).head(20))